# 전처리 & 머신러닝 통합 개인과제

## Part 1. 고객 데이터 품질 개선

### 실무 시나리오

전자상거래 기업의 CRM팀은 고객별 구매 패턴을 분석하고 구매 가능성이 높은 고객을 선별하려고 합니다.  
그러나 전달받은 원본 고객 데이터에는 잘못된 자료형, 범주 표기 불일치, 결측치, 중복 데이터와 극단값이 포함되어 있어 바로 분석에 사용할 수 없습니다.

CRM팀은 **데이터 분석 담당자**로서 원본 데이터의 품질 문제를 진단하고, 이후 Part 2의 EDA와 Part 3의 구매 예측 모델링에 사용할 수 있는 분석용 데이터셋을 만들어야 합니다.

### 과제 목표

- 데이터의 구조·자료형·기초 분포를 확인하고 주요 품질 문제를 파악합니다.
- 결측치, 중복값, 이상치와 범주 표기 불일치를 분석 목적에 맞게 처리합니다.
- 조건 기반 데이터 추출과 파생변수 생성을 수행합니다.
- 전처리 결과를 검증하고 다음 Part에서 사용할 CSV 파일로 저장합니다.

### 사용 환경 및 데이터

- Python 3.X
- pandas, numpy
- `전자상거래_고객구매_원본데이터.csv`

### 제출 결과물

- Part 1 실습 노트북
- `전자상거래_고객구매_전처리완료.csv`

> 문제 1~3은 필수 문제입니다.

## 문제 1. 원본 고객 데이터 품질 진단

### 업무 상황

CRM팀에 분석 일정을 공유하기 전에 원본 데이터가 실제 분석에 사용할 수 있는 상태인지 확인해야 합니다.  
데이터를 수정하기 전에 구조와 품질 문제를 먼저 점검하고, 이후 처리해야 할 항목을 정리하세요.

### 요구사항

1. 원본 데이터를 불러오고 데이터의 크기, 컬럼, 자료형과 기초 통계량을 확인하세요.
2. 컬럼별 결측치와 전체 행 기준 중복 데이터를 확인하세요.
3. 주요 범주형 컬럼의 값을 확인하여 표기 불일치 여부를 파악하세요.
4. 이후 정제가 필요한 주요 품질 문제를 간단히 정리하세요.

### 힌트

- 데이터 구조와 기초 통계량을 함께 확인하면 자료형 오류와 비정상 범위를 찾기 쉽습니다.
- 범주형 컬럼은 고유값을 확인하여 대소문자, 공백, 한글·영문 혼용 여부를 살펴볼 수 있습니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("전자상거래_고객구매_원본데이터.csv")

# TODO: 원본 데이터를 불러와 df에 저장하세요.
df = pd.read_csv(DATA_PATH)

# TODO: 데이터의 크기, 컬럼, 자료형과 기초 통계량을 확인하세요.
print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.describe())

# TODO: 결측치, 중복 데이터와 주요 범주형 컬럼의 값을 확인하세요.
print(df.isna().sum())
print(df.duplicated().sum())

print(df["Gender"].value_counts())
print(df["Region"].value_counts())
print(df["MembershipLevel"].value_counts())
print(df["PreferredCategory"].value_counts())

(1560, 15)
Index(['CustomerID', 'Gender', 'Age', 'Region', 'MembershipLevel',
       'VisitCount', 'AveragePurchaseAmount', 'TotalPurchaseAmount',
       'SatisfactionScore', 'LastPurchaseDate', 'CouponUsed',
       'PreferredCategory', 'SignupDate', 'Email', 'PurchaseStatus'],
      dtype='str')
CustomerID                   str
Gender                       str
Age                          str
Region                       str
MembershipLevel              str
VisitCount                 int64
AveragePurchaseAmount        str
TotalPurchaseAmount      float64
SatisfactionScore        float64
LastPurchaseDate             str
CouponUsed                   str
PreferredCategory            str
SignupDate                   str
Email                        str
PurchaseStatus             int64
dtype: object
        VisitCount  TotalPurchaseAmount  SatisfactionScore  PurchaseStatus
count  1560.000000         1.560000e+03        1521.000000     1560.000000
mean      9.521795         4.841402e+05    

### 품질 진단 결과

- 자료형 정리가 필요한 컬럼: Age, AveragePurchaseAmount, LastPurchaseDate, CouponUsed, SignupDate
- 표기 통일이 필요한 컬럼: Gender, Region, MembershipLevel, PreferredCategory
- 결측치가 있는 컬럼: Gender, Age, Region, MembershipLevel, AveragePurchaseAmount, SatisfactionScore, LastPurchaseDate, PreferredCategory
- 중복 데이터 확인 결과: 전체 행 기준 중복 데이터가 60개 확인됨
- 이상치 확인이 필요한 컬럼: VisitCount, TotalPurchaseAmount
- 이후 처리할 주요 항목: 자료형 변환, 범주 표기 통일, 결측치 및 중복 데이터 처리, 이상치 확인 및 처리

## 문제 2. 분석 가능한 고객 데이터로 정제

### 업무 상황

품질 진단 결과를 바탕으로 고객 데이터를 분석 가능한 상태로 정리해야 합니다.  
처리 과정에서 고객의 실제 구매 행동 정보가 불필요하게 손실되지 않도록 데이터 특성을 고려하세요.

### 요구사항

1. 원본 데이터를 보존한 상태에서 정제용 데이터를 생성하세요.
2. 분석에 맞지 않는 수치형·날짜형 자료형과 범주 표기를 정리하세요.
3. 결측치와 전체 행 기준 중복 데이터를 처리하세요.
4. 주요 수치형 컬럼의 이상치를 탐지하고 적절한 방법으로 처리하세요.
5. 처리 전후의 결측치, 중복값과 이상치 상태를 확인하세요.

### 힌트

- 변환할 수 없는 문자열은 결측치로 바꾼 뒤 일관되게 처리할 수 있습니다.
- 이상치는 무조건 삭제하기보다 실제 우수 고객의 행동일 가능성도 고려하세요.
- IQR은 이상치 후보를 확인하는 대표적인 방법입니다.

In [3]:
# TODO: 원본을 보존하고 정제용 데이터 df_clean을 생성하세요.
df_clean = df.copy()

# TODO: 수치형·날짜형 자료형과 범주 표기를 정리하세요.
#수치
for column in ["Age", "AveragePurchaseAmount"]:
    converted = []

    for value in df_clean[column]:
        try:
            converted.append(float(value))
        except:
            converted.append(np.nan)

    df_clean[column] = converted

#날짜
for column in ["LastPurchaseDate", "SignupDate"]:
    converted = []

    for value in df_clean[column]:
        try:
            converted.append(pd.to_datetime(value))
        except:
            converted.append(np.nan)

    df_clean[column] = converted

#범주 표기
df_clean["Gender"] = df_clean["Gender"].replace({
    " FEMALE ": "Female",
    " MALE ": "Male",
    "female": "Female",
    "male": "Male",
    "F": "Female",
    "M": "Male"
})

df_clean["Region"] = df_clean["Region"].replace({
    " GYEONGGI ": "Gyeonggi",
    " Seoul ": "Seoul",
    " Busan ": "Busan",
    "gyeonggi": "Gyeonggi",
    "seoul": "Seoul",
    "busan": "Busan"
})

df_clean["MembershipLevel"] = df_clean["MembershipLevel"].replace({
    " Basic ": "Basic",
    " Gold ": "Gold",
    "basic": "Basic",
    "silver": "Silver",
    "SILVER": "Silver",
    "gold": "Gold"
})

df_clean["PreferredCategory"] = df_clean["PreferredCategory"].replace({
    " Beauty ": "Beauty",
    " Electronics ": "Electronics",
    "beauty": "Beauty",
    "fashion": "Fashion",
    "FASHION": "Fashion",
    "electronics": "Electronics"
})

df_clean["CouponUsed"] = df_clean["CouponUsed"].replace({
    " n ": 0,
    " y ": 1,
    "N": 0,
    "No": 0,
    "Y": 1,
    "Yes": 1
})

print(df_clean["Gender"].unique())
print("\n", df_clean["Region"].unique())
print("\n", df_clean["MembershipLevel"].unique())
print("\n", df_clean["PreferredCategory"].unique())
print("\n", df_clean["CouponUsed"].unique())

C:\Users\hryug\AppData\Local\Temp\ipykernel_34288\3974947722.py:23: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  converted.append(pd.to_datetime(value))


<StringArray>
['Female', 'Male', 'Other', nan]
Length: 4, dtype: str

 <StringArray>
[nan, 'Gyeonggi', 'Gwangju', 'Daegu', 'Daejeon', 'Busan', 'Seoul', 'Incheon']
Length: 8, dtype: str

 <StringArray>
['Silver', 'Basic', 'VIP', 'Gold', nan]
Length: 5, dtype: str

 <StringArray>
['Beauty', 'Home', 'Fashion', 'Food', 'Electronics', 'Sports', nan]
Length: 7, dtype: str

 [0 1]


In [4]:
# TODO: 결측치와 전체 행 기준 중복 데이터를 처리하세요.
print("처리 전 결측치 \n", df_clean.isna().sum())
print("\n 처리 전 중복:", df_clean.duplicated().sum())

#수치
df_clean["Age"] = df_clean["Age"].fillna(
    df_clean["Age"].median())

df_clean["AveragePurchaseAmount"] = df_clean["AveragePurchaseAmount"].fillna(
    df_clean["AveragePurchaseAmount"].median()
)

df_clean["SatisfactionScore"] = df_clean["SatisfactionScore"].fillna(
    df_clean["SatisfactionScore"].median()
)

#날짜
df_clean["LastPurchaseDate"] = df_clean["LastPurchaseDate"].fillna(
    df_clean["LastPurchaseDate"].median()
)

df_clean["Age"] = df_clean["Age"].astype(int)

df_clean = df_clean.drop_duplicates()

#범주형
for column in ["Gender", "Region", "MembershipLevel", "PreferredCategory"]:
    df_clean[column] = df_clean[column].fillna(
        df_clean[column].mode()[0]
    )

# TODO: 처리 전후 결과를 확인하세요.
print("처리 후 결측치 \n", df_clean.isna().sum())
print("\n 처리 후 중복:", df_clean.duplicated().sum())


처리 전 결측치 
 CustomerID                0
Gender                   26
Age                      58
Region                   31
MembershipLevel          22
VisitCount                0
AveragePurchaseAmount    59
TotalPurchaseAmount       0
SatisfactionScore        39
LastPurchaseDate         30
CouponUsed                0
PreferredCategory        26
SignupDate                0
Email                     0
PurchaseStatus            0
dtype: int64

 처리 전 중복: 60
처리 후 결측치 
 CustomerID               0
Gender                   0
Age                      0
Region                   0
MembershipLevel          0
VisitCount               0
AveragePurchaseAmount    0
TotalPurchaseAmount      0
SatisfactionScore        0
LastPurchaseDate         0
CouponUsed               0
PreferredCategory        0
SignupDate               0
Email                    0
PurchaseStatus           0
dtype: int64

 처리 후 중복: 0


In [5]:
def get_iqr_bounds(series):
    # TODO: Q1, Q3, IQR, 하한과 상한을 반환하세요.
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    return Q1, Q3, IQR, lower_bound, upper_bound

# TODO: 주요 수치형 컬럼의 이상치를 탐지하고 처리하세요.
for column in ["Age", "VisitCount", "AveragePurchaseAmount", "TotalPurchaseAmount", "SatisfactionScore"]:
    Q1, Q3, IQR, lower_bound, upper_bound = get_iqr_bounds(
        df_clean[column]
    )

    outliers = df_clean[
        (df_clean[column] < lower_bound) |
        (df_clean[column] > upper_bound)
    ]

    print(column, "처리 전 이상치:", len(outliers))

    df_clean[column] = df_clean[column].clip(
        lower=lower_bound,
        upper=upper_bound
    )

print("\n")

# TODO: 처리 전후 이상치 상태를 확인하세요.
for column in ["Age", "VisitCount", "AveragePurchaseAmount", "TotalPurchaseAmount", "SatisfactionScore"]:

    Q1, Q3, IQR, lower_bound, upper_bound = get_iqr_bounds(
        df_clean[column]
    )

    outliers = df_clean[
        (df_clean[column] < lower_bound) |
        (df_clean[column] > upper_bound)
    ]
    
    print(column, "처리 후 이상치:", len(outliers))

Age 처리 전 이상치: 11
VisitCount 처리 전 이상치: 15
AveragePurchaseAmount 처리 전 이상치: 24
TotalPurchaseAmount 처리 전 이상치: 53
SatisfactionScore 처리 전 이상치: 10


Age 처리 후 이상치: 0
VisitCount 처리 후 이상치: 0
AveragePurchaseAmount 처리 후 이상치: 0
TotalPurchaseAmount 처리 후 이상치: 0
SatisfactionScore 처리 후 이상치: 0


## 문제 3. 분석용 변수 생성 및 다음 단계 데이터 준비

### 업무 상황

정제된 데이터를 CRM팀의 고객군 분석과 구매 예측 업무에 활용하려면, 원본 컬럼만으로 확인하기 어려운 고객 특성을 분석용 변수로 표현해야 합니다.  
필요한 고객을 조건에 따라 추출하고, 이후 Part 2와 Part 3에서 활용할 파생변수를 만든 뒤 최종 데이터를 저장하세요.

### 요구사항

1. 정제된 데이터에서 업무적으로 의미 있는 조건을 설정하여 고객 데이터를 추출하세요.
2. 고객 분석에 활용할 파생변수를 **2개 이상** 생성하세요.
3. 생성한 파생변수의 값과 분포가 적절한지 확인하세요.
4. 최종 데이터의 결측치, 중복값, 고객 식별자와 타깃 값을 검증하세요.
5. 전처리 완료 데이터를 `전자상거래_고객구매_전처리완료.csv`로 저장하세요.

### 힌트

- 연령대, 구매 수준, 최근 구매 여부, 우수 고객 여부 등을 파생변수 후보로 고려할 수 있습니다.
- 조건 추출 결과는 별도 DataFrame으로 확인해도 되며, 최종 데이터 전체를 삭제할 필요는 없습니다.

- 고객 연령을 구간화한 `AgeGroup`을 생성하세요.
- 총구매금액을 기준으로 `PurchaseGrade`를 생성하세요.

In [6]:
REFERENCE_DATE = pd.Timestamp("2026-06-01")

# TODO: 업무적으로 의미 있는 조건을 설정하여 고객 데이터를 추출하세요.
recent_customers = df_clean[
    (df_clean["LastPurchaseDate"] >= pd.Timestamp("2026-05-01")) &
    (df_clean["LastPurchaseDate"] <= REFERENCE_DATE)
]

print(recent_customers.shape)

# TODO: 분석용 파생변수를 2개 이상 생성하세요.
#1. 연령대
def age_group(age):
    if age < 30:
        return "20대 이하"
    elif age < 40:
        return "30대"
    elif age < 50:
        return "40대"
    elif age < 60:
        return "50대"
    else:
        return "60대 이상"

df_clean["AgeGroup"] = df_clean["Age"].apply(age_group)

#2. 총구매금액
Q1 = df_clean["TotalPurchaseAmount"].quantile(0.25)
Q3 = df_clean["TotalPurchaseAmount"].quantile(0.75)

def purchase_grade(amount):
    if amount >= Q3:
        return "높음"
    elif amount >= Q1:
        return "중간"
    else:
        return "낮음"

df_clean["PurchaseGrade"] = df_clean["TotalPurchaseAmount"].apply(
    purchase_grade
)

# TODO: 파생변수의 값과 분포를 확인하세요.
display(df_clean[["Age", "AgeGroup",
                "TotalPurchaseAmount", "PurchaseGrade"]].head())

print(df_clean["AgeGroup"].value_counts())
print("\n", df_clean["PurchaseGrade"].value_counts())

(419, 15)


,Age,AgeGroup,TotalPurchaseAmount,PurchaseGrade
0,40.0,40대,139800.0,낮음
1,39.0,30대,159200.0,낮음
2,69.5,60대 이상,1055800.0,높음
3,44.0,40대,413200.0,중간
4,37.0,30대,576000.0,높음


AgeGroup
30대       524
40대       463
20대 이하    269
50대       190
60대 이상     54
Name: count, dtype: int64

 PurchaseGrade
중간    750
낮음    375
높음    375
Name: count, dtype: int64


In [7]:
OUTPUT_PATH = Path("전자상거래_고객구매_전처리완료.csv")

# TODO: 최종 데이터 품질을 검증하세요.
print("최종 결측치\n", df_clean.isna().sum())
print("\n 최종 중복:", df_clean.duplicated().sum())

print("\n CustomerID 결측치:", df_clean["CustomerID"].isna().sum())
print("\n CustomerID 중복:", df_clean["CustomerID"].duplicated().sum())

print("\n PurchaseStatus 분포:", df_clean["PurchaseStatus"].value_counts())

# TODO: 전처리 완료 데이터를 CSV 파일로 저장하세요.
df_clean.to_csv(OUTPUT_PATH, index=False)

최종 결측치
 CustomerID               0
Gender                   0
Age                      0
Region                   0
MembershipLevel          0
VisitCount               0
AveragePurchaseAmount    0
TotalPurchaseAmount      0
SatisfactionScore        0
LastPurchaseDate         0
CouponUsed               0
PreferredCategory        0
SignupDate               0
Email                    0
PurchaseStatus           0
AgeGroup                 0
PurchaseGrade            0
dtype: int64

 최종 중복: 0

 CustomerID 결측치: 0

 CustomerID 중복: 0

 PurchaseStatus 분포: PurchaseStatus
0    767
1    733
Name: count, dtype: int64


# 문제 4. 데이터 정제 및 파생변수 설계 근거 설명

## 업무 상황

CRM팀은 전처리 결과가 단순히 실행되는 것뿐 아니라,
적용한 처리 기준과 파생변수가 실제 분석 목적에 적합한지 확인하려고 합니다.

문제 1~3에서 수행한 결과를 바탕으로 다음 내용을 설명하세요.

## 요구사항

### 4-1. 데이터 품질 문제의 영향과 한계

1. 문제 1에서 확인한 주요 품질 문제를 한 가지 이상 선택하세요.
2. 해당 문제가 이후 EDA 또는 모델링 결과에 미칠 수 있는 영향을 설명하세요.
3. 현재 데이터만으로 판단하기 어려운 점이나 추가 확인이 필요한 조건을 작성하세요.

### 4-2. 결측치·이상치 처리 방법의 선택 근거

1. 적용한 결측치 처리 방법을 한 가지 이상 제시하세요.
2. 해당 방법을 선택한 이유를 데이터 특성과 연결하여 작성하세요.
3. 적용한 이상치 처리 방법과 처리 시 주의할 점을 설명하세요.

### 4-3. 파생변수의 활용 의도와 한계

1. `파생변수1`과 `파생변수2`의 생성 기준을 설명하세요.
2. 각 파생변수가 Part 2 고객 특성 분석에서 어떻게 활용될 수 있는지 작성하세요.
3. 파생변수 사용 시 발생할 수 있는 정보 손실 또는 해석상의 한계를 설명하세요.

## 문제 4 결과 작성

### 4-1. 데이터 품질 문제의 영향과 한계

- 선택한 품질 문제:
- EDA 또는 모델링에 미치는 영향:
- 추가 확인이 필요한 조건 또는 한계:

### 4-2. 결측치·이상치 처리 방법의 선택 근거

- 결측치 처리 방법:
- 해당 방법을 선택한 이유:
- 이상치 처리 방법:
- 이상치 처리 시 주의점:

### 4-3. 파생변수의 활용 의도와 한계

- `AgeGroup` 생성 기준과 활용 목적:
- `PurchaseGrade` 생성 기준과 활용 목적:
- 파생변수 사용 시 한계: